# Get IBKR Position

This notebook pulls live IBKR portfolio positions and enriches them with curated security-reference metadata plus fresh IBKR contract details.

Prerequisites:
- Run it with the `py313` kernel.
- Start TWS or IB Gateway locally.
- Enable API access and confirm the host, port, and client ID match your local settings.
- Each live notebook or script needs a unique `IB_CLIENT_ID`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 123
IB_ACCOUNT = "U2935967"

SELECTED_SYMBOL = ""

print(f"Project root: {project_root}")
print(f"IBKR connection: host={IB_HOST}, port={IB_PORT}, client_id={IB_CLIENT_ID}, account={IB_ACCOUNT or '<default>'}")
print(f"Selected symbol filter: {SELECTED_SYMBOL or '<first position>'}")

Project root: /Users/kelvin/git_projects/market_helper
IBKR connection: host=127.0.0.1, port=7497, client_id=123, account=U2935967
Selected symbol filter: <first position>


In [2]:
from IPython.display import display
import pandas as pd

from market_helper.domain.portfolio_monitor import build_live_ibkr_position_security_table

enriched_rows = build_live_ibkr_position_security_table(
    host=IB_HOST,
    port=IB_PORT,
    client_id=IB_CLIENT_ID,
    account_id=IB_ACCOUNT or None,
)

print(
    f"Pulled {len(enriched_rows)} portfolio rows with position metrics, "
    "security-reference enrichment, IBKR contract details, and cash balances when available."
)

enriched_positions = pd.DataFrame(enriched_rows)

if enriched_positions.empty:
    display(enriched_positions)
else:
    preview_columns = [
        "account",
        "internal_id",
        "symbol",
        "local_symbol",
        "quantity",
        "market_value",
        "latest_price",
        "unrealized_pnl",
        "security_mapping_status",
        "security_display_ticker",
        "security_display_name",
        "security_report_category",
        "security_risk_bucket",
        "ibkr_cash_tag",
        "ibkr_cash_conversion_mode",
        "ibkr_cash_source_currencies",
        "contract_long_name",
        "contract_primary_exchange",
    ]
    available_columns = [column for column in preview_columns if column in enriched_positions.columns]
    display(
        enriched_positions.loc[:, available_columns]
        .sort_values("market_value", ascending=False, na_position="last")
        .reset_index(drop=True)
    )

Pulled 27 portfolio rows with position metrics, security-reference enrichment, IBKR contract details, and cash balances when available.


,account,internal_id,symbol,local_symbol,quantity,market_value,latest_price,unrealized_pnl,security_mapping_status,security_display_ticker,security_display_name,security_report_category,security_risk_bucket,ibkr_cash_tag,ibkr_cash_conversion_mode,ibkr_cash_source_currencies,contract_long_name,contract_primary_exchange
0,U2935967,FUT:ZT:CBOT,ZT,ZTM6,6.000000,1.243406e+06,103.617188,-10978.690800,mapped,ZTW00:CBOT,2Y TF,FI,FI,None,None,None,2 Year US Treasury Note,
1,U2935967,FUT:ZF:CBOT,ZF,ZFM6,5.000000,5.397656e+05,107.953125,-7508.289000,mapped,ZFW00:CBOT,5Y TF,FI,FI,None,None,None,5 Year US Treasury Note,
2,U2935967,FUT:AUD:CME,AUD,6AM6,6.000000,4.135986e+05,0.689331,-9392.523800,mapped,AUD:FX_FUT,AUD FX Future,MACRO,MACRO,None,None,None,Australian dollar,
3,U2935967,FUT:EUR:CME,EUR,6EM6,1.000000,1.446188e+05,1.156950,-3533.942300,mapped,EUR:FX_FUT,EUR FX Future,MACRO,MACRO,None,None,None,European Monetary Union Euro,
4,U2935967,FUT:ZN:CBOT,ZN,ZNM6,1.000000,1.107500e+05,110.750000,-2462.757800,mapped,ZNW00:CBOT,10Y TF,FI,FI,None,None,None,10 Year US Treasury Note,
5,U2935967,STK:VOO:ARCA,VOO,VOO,170.000000,1.025097e+05,602.997986,-2360.246901,mapped,VOO,US,DMEQ,EQ,None,None,None,VANGUARD S&P 500 ETF,ARCA
6,U2935967,STK:GLD:ARCA,GLD,GLD,200.000000,8.595640e+04,429.782013,1418.680000,mapped,GLD,Gold,GOLD,GOLD,None,None,None,SPDR GOLD SHARES,ARCA
7,U2935967,STK:ACWD:LSEETF,ACWD,ACWD,300.000000,8.487300e+04,282.910004,-4943.923605,mapped,LON:ACWD,ACWI,EQ,EQ,None,None,None,SS SPD MS AL CO WO UC ET-USD,LSEETF
8,U2935967,FUT:GBP:CME,GBP,6BM6,1.000000,8.256813e+04,1.321090,-1878.312300,mapped,GBP:FX_FUT,GBP FX Future,MACRO,MACRO,None,None,None,British pound,
9,U2935967,STK:IEUR:ARCA,IEUR,IEUR,1000.000000,7.095350e+04,70.953499,6910.696000,mapped,IEUR,EU,DMEQ,EQ,None,None,None,ISHARES CORE MSCI EUROPE ETF,ARCA


In [3]:
if enriched_positions.empty or "symbol" not in enriched_positions.columns:
    print("No enriched portfolio rows were returned.")
    display(enriched_positions)
else:
    if SELECTED_SYMBOL:
        selected_rows = enriched_positions.loc[enriched_positions["symbol"].eq(SELECTED_SYMBOL)]
        if selected_rows.empty:
            print(f"No combined row matched SELECTED_SYMBOL={SELECTED_SYMBOL}; showing the first portfolio row instead.")
            display(enriched_positions.head(1).T)
        else:
            display(selected_rows.head(1).T)
    else:
        display(enriched_positions.head(1).T)

    cash_positions = enriched_positions.iloc[0:0]
    cash_mask = pd.Series(False, index=enriched_positions.index)
    if "security_universe_type" in enriched_positions.columns:
        cash_mask = cash_mask | enriched_positions["security_universe_type"].fillna("").eq("CASH")
    if "ibkr_sec_type" in enriched_positions.columns:
        cash_mask = cash_mask | enriched_positions["ibkr_sec_type"].fillna("").eq("CASH")
    cash_positions = enriched_positions.loc[cash_mask]
    print(f"Cash rows: {len(cash_positions)}")
    if not cash_positions.empty:
        cash_columns = [
            "account",
            "internal_id",
            "symbol",
            "local_symbol",
            "quantity",
            "market_value",
            "latest_price",
            "security_display_name",
            "security_mapping_status",
            "ibkr_cash_tag",
            "ibkr_cash_conversion_mode",
            "ibkr_cash_source_currencies",
        ]
        available_columns = [column for column in cash_columns if column in cash_positions.columns]
        display(
            cash_positions.loc[:, available_columns]
            .sort_values("market_value", ascending=False, na_position="last")
            .reset_index(drop=True)
        )

    if "security_mapping_status" in enriched_positions.columns:
        unmapped_positions = enriched_positions.loc[
            enriched_positions["security_mapping_status"].fillna("").ne("mapped")
        ]
        print(f"Unmapped or partially mapped rows: {len(unmapped_positions)}")
        if not unmapped_positions.empty:
            diagnostic_columns = [
                "account",
                "symbol",
                "local_symbol",
                "internal_id",
                "security_mapping_status",
                "contract_primary_exchange",
            ]
            available_columns = [column for column in diagnostic_columns if column in unmapped_positions.columns]
            display(unmapped_positions.loc[:, available_columns].reset_index(drop=True))


,0
as_of,2026-04-03T12:56:25+00:00
account,U2935967
internal_id,STK:ACWD:LSEETF
con_id,91812967
symbol,ACWD
...,...
security_price_source_provider,google_finance
security_price_source_symbol,LON:ACWD
security_fx_source_provider,google_finance
security_fx_source_symbol,CURRENCY:GBPUSD


Cash rows: 1


,account,internal_id,symbol,local_symbol,quantity,market_value,latest_price,security_display_name,security_mapping_status,ibkr_cash_tag,ibkr_cash_conversion_mode,ibkr_cash_source_currencies
0,U2935967,CASH:SGD_CASH_VALUE:MANUAL,SGD_CASH_VALUE,SGD Cash,-1131.793982,-1131.793982,1.0,Cash,mapped,TOTALCASHBALANCE_CONVERTED_TO_SGD,multi_currency_to_sgd,"AUD,CNH,EUR,GBP,SGD,USD"


Unmapped or partially mapped rows: 3


,account,symbol,local_symbol,internal_id,security_mapping_status,contract_primary_exchange
0,U2935967,SPY,SPY 260618P00620000,OUTSIDE_SCOPE:OPT:SPY:AMEX,outside_scope,
1,U2935967,SPY,SPY 260918P00590000,OUTSIDE_SCOPE:OPT:SPY:AMEX,outside_scope,
2,U2935967,VXM,VXMM6,FUT:VXM:CFE,unmapped,
